# 04 — Hedge Benefit Ratio and Bucket Capital

**Hedge benefit ratio** (HBR) for a bucket:

```
HBR = aggregate_net_long / (aggregate_net_long + aggregate_net_short)
```

If there are no net shorts after netting, HBR = 1.0 and the bucket has no hedge
benefit reduction.

**Bucket DRC capital** (risk-weighted, floored at zero):

```
capital = max( Σ(RW * long_net) - HBR * Σ(RW * short_net), 0 )
```

Risk weights: NON\_US\_SOVEREIGN IG=0.6 %, SG=22 %, sub-SG=50 %;
PSE\_GSE IG=2.1 %, SG=22 %, sub-SG=50 %; CORPORATE IG=4.1 %, SG=22 %,
sub-SG=50 %; DEFAULTED=100 %.

*Regulatory refs*: Basel MAR22.17-19; US NPR § 210(a)(2)(iv)

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Full pipeline to bucket capital

In [ ]:
from frtb_drc.scaffold import calculate_drc_capital

result = calculate_drc_capital(positions, context=context)
cat = result.categories[0]

rows = [
    [
        b.bucket_key,
        f"{b.hbr.aggregate_net_long:>14,.2f}",
        f"{b.hbr.aggregate_net_short:>14,.2f}",
        f"{b.hbr.ratio:.6f}",
        f"{b.weighted_long:>14,.2f}",
        f"{b.weighted_short:>14,.2f}",
        f"{b.capital:>14,.2f}",
        "Y" if b.floor_applied else "",
    ]
    for b in cat.bucket_results
]
display(markdown_table(
    ["Bucket", "Agg net long", "Agg net short", "HBR", "W-long", "W-short", "Capital", "Floor?"],
    rows,
))

## HBR mechanics

### CORPORATE — partial hedge

`eta-finance` and `zeta-metals` have REJECTED shorts that survive as net short
positions.  After risk-weighting:

- `weighted_short > 0` → HBR < 1.0
- Capital = weighted\_long - HBR * weighted\_short

The HBR ensures the hedge benefit is shared proportionally across longs and shorts.

In [ ]:
corp = next(b for b in cat.bucket_results if b.bucket_key == "CORPORATE")
print(f"CORPORATE")
print(f"  aggregate_net_long  = {corp.hbr.aggregate_net_long:,.2f}")
print(f"  aggregate_net_short = {corp.hbr.aggregate_net_short:,.2f}")
print(f"  HBR                 = {corp.hbr.ratio:.6f}")
print(f"  weighted_long       = {corp.weighted_long:,.2f}")
print(f"  weighted_short      = {corp.weighted_short:,.2f}")
print(f"  capital (unfloored) = {corp.weighted_long - corp.hbr.ratio * corp.weighted_short:,.2f}")
print(f"  capital (final)     = {corp.capital:,.2f}")

### NON\_US\_SOVEREIGN — no net shorts after netting

UK, Japan, and Brazil shorts were fully consumed by their respective longs during netting.
No net short positions remain.  HBR = 1.0, but `weighted_short = 0` — so HBR has
no practical effect.

In [ ]:
sov = next(b for b in cat.bucket_results if b.bucket_key == "NON_US_SOVEREIGN")
print(f"NON_US_SOVEREIGN")
print(f"  aggregate_net_long  = {sov.hbr.aggregate_net_long:,.2f}")
print(f"  aggregate_net_short = {sov.hbr.aggregate_net_short:,.2f}")
print(f"  HBR                 = {sov.hbr.ratio:.6f}")
print(f"  capital             = {sov.capital:,.2f}")

## Bucket capital waterfall

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [b.bucket_key for b in cat.bucket_results]
w_longs = [b.weighted_long / 1e6 for b in cat.bucket_results]
w_shorts_reduced = [b.hbr.ratio * b.weighted_short / 1e6 for b in cat.bucket_results]
capitals = [b.capital / 1e6 for b in cat.bucket_results]

x = np.arange(len(labels))
width = 0.28

fig, ax = plt.subplots(figsize=(9, 4.5))
bars1 = ax.bar(x - width, w_longs, width, label="RW * net long", color="#2563EB", alpha=0.85)
bars2 = ax.bar(x, w_shorts_reduced, width, label="HBR * RW * net short", color="#F87171", alpha=0.85)
bars3 = ax.bar(x + width, capitals, width, label="Bucket capital", color="#059669", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("USD M")
ax.set_title("Bucket capital waterfall: RW * long - HBR * RW * short")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()